# Square-band overlays for saved experience heatmaps

Run these cells in a CPU Colab. The default Drive path is the completed wider one-macro retry. The notebook reads two saved JSON files and writes the plots to the local Colab runtime.

Dashed squares mark the band boundaries around the plotted goal. Orange lines highlight the inner and outer edges of the panel's band. Boundaries outside the plot are clipped. The squares mark start bands for collection goals and serve as distance references for held-out goals. Observations can come from anywhere along a successful path. Each panel uses its own unclipped cost scale. Set `HEATMAP_GOALS` to view other goals.

In [ ]:
from google.colab import drive
from pathlib import Path
import json

drive.mount('/content/drive')
RUN = (Path('/content/drive/MyDrive/datasets/curriculum_cost_adapter')
       / 'experience_one_macro_adapter_wide_v1' / '20261108T091827483705Z')
PLOT_DIR = Path('/content/experience_band_heatmaps')
report = json.loads((RUN / 'comparison.json').read_text())
manifest = json.loads((RUN / 'experiment.json').read_text())['manifest']

In [ ]:
from IPython.display import Image, display
from matplotlib import patheffects
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import numpy as np

HEATMAP_GOALS = [0, 15]
grid = report['grid']
shape = (len(grid['ys']), len(grid['xs']))
methods = list(report['methods'])
bands = manifest['bands']
collection_goals = {tuple(case['goal']) for stage in manifest['stages'] for case in stage}
PLOT_DIR.mkdir(parents=True, exist_ok=True)

for goal_id in HEATMAP_GOALS:
    goal = tuple(report['methods']['raw'][goal_id]['goal'])
    gx, gy = goal
    cols = 5
    rows = (len(methods) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(17, 4.8 * rows),
                             squeeze=False, constrained_layout=True)
    for ax in axes.flat[len(methods):]:
        ax.set_visible(False)
    for ax, method in zip(axes.flat, methods):
        row = report['methods'][method][goal_id]
        costs = np.asarray(row['adapted_costs'])[grid['point_ids']].reshape(shape)
        mesh = ax.pcolormesh(grid['xs'], grid['ys'], costs, shading='nearest', cmap='viridis',
                             vmin=0, vmax=max(float(costs.max()), 1e-9))
        active = ()
        title = {'raw': 'Raw MSE', 'handcrafted': 'Hand-crafted'}.get(method, method)
        if method.startswith('band_'):
            number = int(method.split('_')[1])
            active = bands[number - 1]
            title = f'Band {number} · {active[0]}–{active[1]}'
            if report['checkpoints'][method]['updates'] == 0:
                title += '\nCarried forward (0 updates)'
        for _, radius in bands:
            highlight = radius in active
            width = 1.6 if highlight else 0.8
            outline = Rectangle((gx - radius, gy - radius), 2 * radius, 2 * radius,
                                fill=False, edgecolor='#ffaf36' if highlight else 'white',
                                linestyle='-' if highlight else '--', linewidth=width,
                                zorder=4 if highlight else 3,
                                path_effects=[patheffects.withStroke(linewidth=width + 0.9,
                                                                      foreground='#282838')])
            ax.add_patch(outline)
        ax.scatter(*goal, marker='*', s=110, c='black', edgecolor='white', zorder=5)
        ax.set(title=title, xlabel='Ball x', xlim=(35, 60), ylim=(4, 60), aspect='equal')
        fig.colorbar(mesh, ax=ax, shrink=.6, label='Latent MSE')
    for ax in axes[:, 0]:
        ax.set_ylabel('Ball y')
    label = 'Collection start bands' if goal in collection_goals else 'Held-out goal · reference bands'
    fig.suptitle(f'Goal {goal_id} {goal} · {label}')
    fig.legend(handles=[Line2D([], [], color='#4e4e5e', linestyle='--', label='Square band boundaries'),
                        Line2D([], [], color='#d57900', label='Inner and outer edges of panel band')],
               loc='upper center', bbox_to_anchor=(.5, -.005), ncol=2, frameon=False)
    path = PLOT_DIR / f'goal_{goal_id}_bands.png'
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=str(path)))